# COMP3710 Lab 2 - Part 2: Eigenfaces

This notebook contains the teacher-provided Part 2 code from the lab sheet.

## Task overview

1. Load the Labeled Faces in the Wild (LFW) dataset.
2. Split the data into training and testing sets.
3. Centre the data and compute PCA/eigenfaces using NumPy SVD.
4. Project the data into the learned face space.
5. Plot the eigenfaces and the compactness curve.
6. Train and evaluate a Random Forest face classifier.

In [ ]:
from sklearn.datasets import fetch_lfw_people
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np

# Download the data, if not already on disk and load it as numpy arrays
lfw_people = fetch_lfw_people(min_faces_per_person=70, resize=0.4)

In [ ]:
# Introspect the images arrays to find the shapes (for plotting)
n_samples, h, w = lfw_people.images.shape

# For machine learning we use the 2 data directly (as relative pixel
# positions info is ignored by this model)
X = lfw_people.data
n_features = X.shape[1]

# The label to predict is the id of the person
y = lfw_people.target
target_names = lfw_people.target_names
n_classes = target_names.shape[0]

print("Total dataset size:")
print("n_samples: %d" % n_samples)
print("n_features: %d" % n_features)
print("n_classes: %d" % n_classes)

In [ ]:
# Split into a training set and a test set using a stratified k fold
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Compute a PCA (eigenfaces) on the face dataset (treated as unlabeled
# dataset): unsupervised feature extraction / dimensionality reduction
n_components = 150

In [ ]:
# Center data
mean = np.mean(X_train, axis=0)
X_train -= mean
X_test -= mean

# Eigen-decomposition
U, S, V = np.linalg.svd(X_train, full_matrices=False)
components = V[:n_components]
eigenfaces = components.reshape((n_components, h, w))

# Project into PCA subspace
X_transformed = np.dot(X_train, components.T)
print(X_transformed.shape)

X_test_transformed = np.dot(X_test, components.T)
print(X_test_transformed.shape)

In [ ]:
import matplotlib.pyplot as plt

# Qualitative evaluation of the predictions using matplotlib
def plot_gallery(images, titles, h, w, n_row=3, n_col=4):
    """Helper function to plot a gallery of portraits."""
    plt.figure(figsize=(1.8 * n_col, 2.4 * n_row))
    plt.subplots_adjust(bottom=0, left=.01, right=.99, top=.90, hspace=.35)
    for i in range(n_row * n_col):
        plt.subplot(n_row, n_col, i + 1)
        plt.imshow(images[i].reshape((h, w)), cmap=plt.cm.gray)
        plt.title(titles[i], size=12)
        plt.xticks(())
        plt.yticks(())

eigenface_titles = ["eigenface %d" % i for i in range(eigenfaces.shape[0])]
plot_gallery(eigenfaces, eigenface_titles, h, w)

plt.show()

In [ ]:
# Evaluate dimensionality reduction using a compactness plot
explained_variance = (S ** 2) / (n_samples - 1)
total_var = explained_variance.sum()
explained_variance_ratio = explained_variance / total_var
ratio_cumsum = np.cumsum(explained_variance_ratio)
print(ratio_cumsum.shape)
eigenvalueCount = np.arange(n_components)

plt.plot(eigenvalueCount, ratio_cumsum[:n_components])
plt.title('Compactness')
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Build random forest
estimator = RandomForestClassifier(
    n_estimators=150, max_depth=15, max_features=150
)
estimator.fit(X_transformed, y_train)  # Expects X as [n_samples, n_features]

predictions = estimator.predict(X_test_transformed)
correct = predictions == y_test
total_test = len(X_test_transformed)

# print("Gnd Truth:", y_test)
print("Total Testing", total_test)
print("Predictions", predictions)
print("Which Correct:", correct)
print("Total Correct:", np.sum(correct))
print("Accuracy:", np.sum(correct) / total_test)

print(classification_report(y_test, predictions, target_names=target_names))

## Completed Part 2 implementation

The teacher-provided code above is preserved unchanged. The following cells contain the separate completed and reproducible implementation.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import fetch_lfw_people
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# Store the downloaded dataset inside the project so the notebook is portable.
data_home = Path("data")
lfw_people_completed = fetch_lfw_people(
    data_home=data_home,
    min_faces_per_person=70,
    resize=0.4,
)

n_samples_completed, h_completed, w_completed = lfw_people_completed.images.shape
X_completed = lfw_people_completed.data.copy()
y_completed = lfw_people_completed.target
target_names_completed = lfw_people_completed.target_names
n_features_completed = X_completed.shape[1]
n_classes_completed = target_names_completed.shape[0]

print("Total dataset size:")
print("n_samples:", n_samples_completed)
print("n_features:", n_features_completed)
print("n_classes:", n_classes_completed)


In [ ]:
# Use stratification so every identity keeps a similar class proportion in both sets.
X_train_completed, X_test_completed, y_train_completed, y_test_completed = train_test_split(
    X_completed,
    y_completed,
    test_size=0.25,
    random_state=42,
    stratify=y_completed,
)

n_components_completed = 150

# Centre both sets using only the training mean to prevent test-data leakage.
mean_face_completed = np.mean(X_train_completed, axis=0)
X_train_centered = X_train_completed - mean_face_completed
X_test_centered = X_test_completed - mean_face_completed

# Compute PCA through the SVD of the centred training matrix.
U_completed, S_completed, Vt_completed = np.linalg.svd(
    X_train_centered,
    full_matrices=False,
)
components_completed = Vt_completed[:n_components_completed]
eigenfaces_completed = components_completed.reshape(
    (n_components_completed, h_completed, w_completed)
)

# Project the training and testing sets into the learned face space.
X_train_pca = X_train_centered @ components_completed.T
X_test_pca = X_test_centered @ components_completed.T

print("Training face-space shape:", X_train_pca.shape)
print("Testing face-space shape:", X_test_pca.shape)


In [ ]:
def plot_eigenface_gallery(images, titles, image_height, image_width, n_row=3, n_col=4):
    """Plot the first eigenfaces in a compact gallery."""
    plt.figure(figsize=(1.8 * n_col, 2.4 * n_row))
    plt.subplots_adjust(bottom=0, left=0.01, right=0.99, top=0.90, hspace=0.35)
    for index in range(n_row * n_col):
        plt.subplot(n_row, n_col, index + 1)
        plt.imshow(
            images[index].reshape((image_height, image_width)),
            cmap=plt.cm.gray,
        )
        plt.title(titles[index], size=12)
        plt.xticks(())
        plt.yticks(())


eigenface_titles_completed = [
    f"eigenface {index}" for index in range(eigenfaces_completed.shape[0])
]
plot_eigenface_gallery(
    eigenfaces_completed,
    eigenface_titles_completed,
    h_completed,
    w_completed,
)
plt.show()


In [ ]:
# Calculate cumulative explained variance using the training sample count.
explained_variance_completed = (S_completed ** 2) / (len(X_train_centered) - 1)
explained_variance_ratio_completed = (
    explained_variance_completed / explained_variance_completed.sum()
)
cumulative_variance_completed = np.cumsum(explained_variance_ratio_completed)

component_numbers = np.arange(1, n_components_completed + 1)
plt.figure(figsize=(8, 5))
plt.plot(
    component_numbers,
    cumulative_variance_completed[:n_components_completed],
)
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance ratio")
plt.title("PCA Compactness")
plt.grid(True)
plt.show()

print(
    f"Variance retained by {n_components_completed} components: "
    f"{cumulative_variance_completed[n_components_completed - 1]:.4f}"
)


In [ ]:
# Train a reproducible Random Forest on the PCA face-space features.
random_forest_completed = RandomForestClassifier(
    n_estimators=150,
    max_depth=15,
    max_features=150,
    random_state=42,
    n_jobs=-1,
)
random_forest_completed.fit(X_train_pca, y_train_completed)

predictions_completed = random_forest_completed.predict(X_test_pca)
correct_completed = predictions_completed == y_test_completed
accuracy_completed = np.mean(correct_completed)

print("Total Testing:", len(y_test_completed))
print("Total Correct:", np.sum(correct_completed))
print(f"Accuracy: {accuracy_completed:.4f}")
print()
print(
    classification_report(
        y_test_completed,
        predictions_completed,
        target_names=target_names_completed,
        zero_division=0,
    )
)
